# Đồ án bất động sản & tối ưu hiệu năng
Bùi Kiếm Khoa — n23dcn029; Lương Võ Hân Hân — n23dccn017.

Notebook đọc dữ liệu và số đo đã tạo. Chạy `python run_project.py` từ thư mục gốc nếu chưa có đầu ra. Dữ liệu thật và mô phỏng được tách riêng.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / "run_project.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "run_project.py").exists(), "Mở notebook trong thư mục dự án"
base = pd.read_csv(ROOT / "data/processed/real_estate_clean.csv")
base.head()

## 1. Chất lượng dữ liệu thật
Không thêm giá trị thiếu giả vào mẫu gốc. Đọc audit do pipeline ghi.

In [ ]:
metadata = json.loads((ROOT / "data/processed/source_metadata.json").read_text(encoding="utf-8"))
print(metadata["audit"])
base.describe()

## 2. Khám phá
Giá theo 10.000 TWD/ping; không phải VND hoặc tổng giá căn nhà. Tương quan không phải nhân quả.

In [ ]:
base.plot.scatter(x="mrt_distance_m", y="price_10k_twd_per_ping", alpha=.6)
plt.show()
base[["house_age", "mrt_distance_m", "price_10k_twd_per_ping"]].corr()

## 3. Mức dữ liệu mô phỏng
Chỉ dùng cho thử tải, không phải giao dịch thật bổ sung.

In [ ]:
manifest = pd.DataFrame(json.loads((ROOT / "data/benchmark/manifest.json").read_text(encoding="utf-8")))
manifest

## 4. Kết quả đo
Thời gian trung vị, peak RAM toàn tiến trình; xem cả min/max. Cache hệ điều hành không xóa.

In [ ]:
summary = pd.read_csv(ROOT / "results/summary.csv")
assert summary.correct.all()
summary[["rows", "method", "median_s", "median_peak_mib", "speedup", "correct"]]

In [ ]:
summary.pivot(index="rows", columns="method", values="median_s").plot(marker="o", ylabel="Seconds")
plt.show()

## 5. Đọc các kết quả gốc và giới hạn
Không suy rộng benchmark mô phỏng thành hiệu năng của mọi dữ liệu lớn. Xem `report/bao_cao.html` để đọc đầy đủ phương pháp, nguồn và giới hạn.

In [ ]:
runs = json.loads((ROOT / "results/runs.json").read_text(encoding="utf-8"))
print("Số phép đo:", len(runs))
print("Tất cả khớp:", all(r["correct"] for r in runs))
runs[0]

## 6. Nguồn Việt Nam bổ sung
Dữ liệu tin đăng bên thứ ba, không phải giao dịch xác minh. Số đo nằm riêng trong `results/vietnam/`. Không trộn đơn vị với UCI.

In [ ]:
vn_profile = json.loads((ROOT / "data/vietnam/processed/profile.json").read_text(encoding="utf-8"))
print("Hàng nguồn:", vn_profile["raw_rows"], "Hàng sạch:", vn_profile["clean_rows"])
vn_summary = pd.read_csv(ROOT / "results/vietnam/summary.csv")
assert vn_summary.correct.all()
vn_summary[["rows", "method", "median_s", "median_peak_mib", "speedup"]]

## 7. Chất lượng trước–sau và thí nghiệm mở rộng
Bước chuẩn bị dùng Arrow batch và thứ tự hash xác định; số đo mới không dùng bộ mẫu NumPy của phiên cũ. Đọc báo cáo nâng cấp để hiểu giới hạn RAM và cách tra cứu.

In [ ]:
quality = pd.read_csv(ROOT / "data/vietnam/processed/quality_before_after.csv")
quality

In [ ]:
extra_path = ROOT / "results/vietnam/experiments/expanded_summary.csv"
if extra_path.exists():
    extra = pd.read_csv(extra_path)
    assert extra.correct.all()
    print(extra[["query", "chunksize", "method", "median_s", "median_peak_mib"]].to_string(index=False))

## 8. Kiểm chứng giới hạn bộ nhớ và hòa vốn
Giới hạn commit Windows khác RAM vật lý. Giữ riêng OOM/lỗi native; chỉ tính thời gian ca thành công. Đọc `report/kiem_chung_du_lieu_lon.html` để xem điều kiện đo và đóng góp dữ liệu.

In [ ]:
memory = pd.read_csv(ROOT / "results/vietnam/memory_limits/summary.csv")
memory

In [ ]:
break_even = pd.read_csv(ROOT / "results/vietnam/break_even/break_even.csv")
# N là ước tính từ trung vị; có cả đối chứng DuckDB đọc CSV trực tiếp.
break_even